# Classification: K-Nearest Neighbors (K-NN)

## Justification of Preprocessing Strategy

### **Distance-Based Model Requirements**

K-NN is a non-parametric, distance-based algorithm. It identifies the $k$(n_neighbours) closest training examples to a new point and classifies it based on their majority vote. Because it relies on distance calculations (Euclidean or Manhattan), features with large numerical scales (such as Glucose) would dominate smaller-scale features (such as BMI). Therefore, Standardization and Normalization are essential to place all variables on a comparable scale.

### **The Challenge of 100,000 Samples**

K-NN is a *lazy learner*, meaning it does not build a model during training and instead stores the full dataset. The main computational cost appears at prediction time, when distances are computed against a very large number of training samples. To make this feasible, we use `n_jobs=-1` to parallelize distance calculations.

## Experiment Design

We run a tournament of 6 experiments ($2$ scalers × $3$ optimization levels) to identify the most accurate and efficient configuration:

- **Standardized Baseline**: Default parameters ($k=5$, uniform weights, Euclidean distance).
- **Standardized GridSearchCV**: Exhaustive search over $k$, weights, and distance metric $p$.
- **Standardized Optuna**: Bayesian search to fine-tune $k$ and weights.
- **Normalized Baseline**: Default parameters on a $[0,1]$ scale.
- **Normalized GridSearchCV**: Comparison of hyperparameter performance on normalized data.
- **Normalized Optuna**: Final optimization in the normalized feature space.

In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_KNN")

In [ ]:
df = pd.read_csv("C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/data/diabetes_dataset_new_variables.csv")

# Categorical columns for One-Hot Encoding
categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target (Removing target and stage)
X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Full dataset split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_knn_metrics(y_true, y_pred, duration):
    """Helper function to log metrics consistently across all runs[cite: 1]"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# 6 RUNS 
# ---------------------------------------------------------
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

for s_name, scaler in scalers.items():
    # Execute Feature Scaling (Crucial for KNN distances)
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

    # --- RUN: BASELINE (Default Parameters as per documentation) ---
    with mlflow.start_run(run_name=f"KNN_{s_name}_Baseline"):
        # Defaults: n_neighbors=5, weights='uniform', p=2
        model = KNeighborsClassifier(n_jobs=-1)
        
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        y_pred = model.predict(X_test_scaled)
        
        mlflow.log_params(model.get_params())
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "none")
        log_knn_metrics(y_test, y_pred, duration)

    # --- RUN: GRIDSEARCHCV (Main Parameters) ---
    with mlflow.start_run(run_name=f"KNN_{s_name}_GridSearch"):
        # Focusing on k, weights and p (distance metric)
        param_grid = {
            'n_neighbors': [3, 7, 11, 15],
            'weights': ['uniform', 'distance'],
            'p': [1, 2] # 1: Manhattan, 2: Euclidean
        }
        
        grid = GridSearchCV(
            KNeighborsClassifier(n_jobs=-1), 
            param_grid, cv=5, scoring='recall', n_jobs=-1
        )
        
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        y_pred = grid.best_estimator_.predict(X_test_scaled)
        
        mlflow.log_params(grid.best_params_)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "GridSearchCV")
        log_knn_metrics(y_test, y_pred, duration)

    # --- RUN: OPTUNA (Bayesian Search) ---
    def objective(trial):
        params = {
            "n_neighbors": trial.suggest_int("n_neighbors", 1, 31),
            "weights": trial.suggest_categorical("weights", ["uniform", "distance"]),
            "p": trial.suggest_int("p", 1, 2)
        }
        
        knn_opt = KNeighborsClassifier(**params, n_jobs=-1)
        score = cross_val_score(
            knn_opt, X_train_scaled, y_train, 
            cv=3, scoring='recall', n_jobs=-1
        ).mean()
        return score

    with mlflow.start_run(run_name=f"KNN_{s_name}_Optuna"):
        study = optuna.create_study(direction="maximize")
        start_time = time.time()
        study.optimize(objective, n_trials=10) 
        duration = time.time() - start_time
        
        # Retrain best found model to extract final metrics
        best_knn = KNeighborsClassifier(**study.best_params, n_jobs=-1)
        best_knn.fit(X_train_scaled, y_train)
        
        mlflow.log_params(study.best_params)
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("optimization", "optuna")
        log_knn_metrics(y_test, best_knn.predict(X_test_scaled), duration)

## K-NN Runs Comparison

| Run | Scaler | Optimization | n_neighbors | Weights | p | Accuracy | F1 | Recall | Fit Time (s) |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|
| KNN_Normalization_Baseline | Normalization | none | 5 | uniform | 2 | 0.67310 | 0.73728 | 0.76450 | 0.036 |
| KNN_Normalization_GridSearch | Normalization | GridSearchCV | 15 | uniform | 1 | 0.76740 | 0.81431 | 0.85000 | 195.416|
| KNN_Normalization_Optuna | Normalization | Optuna | 26 | distance | 1 | 0.78065 | 0.82562 | 0.86542 | 302.839 |
| KNN_Standardization_Baseline | Standardization | none | 5 | uniform | 2 | 0.80935 | 0.83984 | 0.83308 | 0.071 |
| KNN_Standardization_GridSearch | Standardization | GridSearchCV | 15 | uniform | 1 | 0.84045 | 0.86570 | 0.85708 | 197.75|
| KNN_Standardization_Optuna | Standardization | Optuna | 23 | uniform | 1 | 0.84550 | 0.87019 | 0.86308 | 340.744 |


### Analysis of Results

- **Best overall metrics**: `KNN_Standardization_Optuna` achieves the highest Accuracy (0.8455) and F1 (0.8702), with Recall 0.8631.
- **Effect of scaling**: Standardization consistently outperforms Normalization for K-NN in these runs (higher Accuracy/F1).
- **Optimization impact**: GridSearchCV and Optuna improve metrics compared to baselines, at the cost of substantially longer fit times (minutes vs fractions of a second).
- **Recall vs compute trade-off**: Normalization+Optuna produced the highest recall (0.8654) but with much lower overall accuracy than the standardized Optuna run. The recall gain is small relative to the loss in Accuracy and F1.

### Justification of the selected run

- **Selected run**: `KNN_Standardization_Optuna`
- **Why this run fits our objective**: our goal is to obtain the best clinical classification performance overall, not only the highest recall. In this context, Accuracy and F1 are more representative of the model's balance between identifying diabetic patients correctly and avoiding too many false alarms.
- **Trade-off assessment**: although `KNN_Normalization_Optuna` has a slightly higher recall, `KNN_Standardization_Optuna` delivers better Accuracy and F1, so it is the stronger choice when the objective is the best global classifier.
- **Practical conclusion**: the selected run gives the most robust balance between predictive quality and operational cost, so it is the most appropriate option for deployment.
